# noise-batch-from-latent — ex1: build (B, latent_dim, 1, 1) spatial-prefix noise for DCGAN G

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `noise-batch-from-latent`. Running the final beacon cell reports progress against the `GAN: Noise batch from latent_dim` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `GAN: Noise batch from latent_dim` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`noise-batch-from-latent`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "noise-batch-from-latent"
DD_SUBTOPIC = "GAN: Noise batch from latent_dim"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## DCGAN noise batch (spatial latent) — quick refresher

The DCGAN generator's input is a 4-D noise tensor, not a flat vector — the spatial dims are 1×1 placeholders that ConvTranspose layers will expand:

```python
noise = t.randn(batch_size, latent_dim, 1, 1, device=device)
fakes = G(noise)   # G is built of ConvTranspose2d layers
```

Shape contract: `(B, latent_dim, 1, 1)`. The trailing 1×1 makes it a valid input to `ConvTranspose2d(latent_dim, ..., kernel_size=4)` — the first transposed conv blows the spatial dims out to 4×4.

**Compared to flat `(B, latent_dim) + view + Linear` (the ARENA form).** Some implementations use `Linear` then `view` to a spatial seed; others go straight from `(B, latent_dim, 1, 1)` through `ConvTranspose`. Both produce identical output shapes; the fully-convolutional form (this recap) is the original DCGAN paper.

**Standard-normal noise, not uniform.** `t.randn` (not `t.rand`) — the prior is `N(0, 1)`. Uniform-noise priors exist but are not the DCGAN default and produce different visual artifacts.

### Exercise 1 — build (B, latent_dim, 1, 1) spatial-prefix noise for DCGAN G

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply `t.randn(B, latent_dim, 1, 1)` to construct the 4-D Gaussian noise batch that DCGAN generators accept as input.
> Keywords: dcgan, noise, randn, spatial-prefix
> ```

**KCs targeted:** `randn-4d-shape`, `spatial-1x1-prefix`

Implement `ex1_dcgan_noise_batch(batch_size, latent_dim, generator)`. The DCGAN generator input — 4-D noise tensor with trailing 1×1 spatial dims:

1. Sample standard-normal noise of shape `(batch_size, latent_dim, 1, 1)`.
2. Use `t.randn(batch_size, latent_dim, 1, 1, generator=generator)` — pass the generator through so the call is reproducible.
3. Return the resulting tensor (dtype `float32`).

Important constraints:
- The output must be 4-D (`x.ndim == 4`), not 2-D. A 2-D flat vector won't pass through a `ConvTranspose2d(latent_dim, ..., kernel_size=4)`.
- The trailing two dims must be EXACTLY 1 — the first transposed conv expects a 1×1 seed.
- Use `t.randn` (not `t.rand`); the DCGAN prior is `N(0, 1)`, not uniform.

Input: `batch_size` int, `latent_dim` int, `generator` `torch.Generator` for seed reproducibility.
Output: `(batch_size, latent_dim, 1, 1)` float32 tensor.

The visualization renders the noise as a `(B × latent_dim)` heatmap (collapsing the 1×1 spatial axes) so you can verify the shape contract.

In [ ]:
def ex1_dcgan_noise_batch(batch_size: int, latent_dim: int, generator: 'torch.Generator') -> Tensor:
    """Standard-normal noise of shape (B, latent_dim, 1, 1)."""
    raise NotImplementedError()


def _test_ex1():
    import torch.nn as nn

    # Shape contract — must be (B, latent_dim, 1, 1).
    rng = t.Generator().manual_seed(0)
    noise = ex1_dcgan_noise_batch(8, 100, rng)
    assert noise.shape == (8, 100, 1, 1), f'expected (8, 100, 1, 1), got {tuple(noise.shape)}'
    assert noise.dtype == t.float32
    assert noise.ndim == 4, f'noise must be 4-D, got ndim={noise.ndim}'

    # Distribution sanity — sample a large batch and check ~N(0, 1).
    rng2 = t.Generator().manual_seed(0)
    big = ex1_dcgan_noise_batch(2000, 100, rng2)
    assert abs(big.mean().item()) < 0.05, f'noise mean should be ~0, got {big.mean().item():.4f}'
    assert abs(big.std().item() - 1.0) < 0.05, f'noise std should be ~1, got {big.std().item():.4f}'

    # Reproducibility — same seed → same noise.
    rng_a = t.Generator().manual_seed(42)
    rng_b = t.Generator().manual_seed(42)
    n_a = ex1_dcgan_noise_batch(4, 8, rng_a)
    n_b = ex1_dcgan_noise_batch(4, 8, rng_b)
    assert t.equal(n_a, n_b), 'same seed must produce same noise'

    # Shape must be valid input to a real DCGAN first layer.
    first_layer = nn.ConvTranspose2d(100, 512, kernel_size=4, stride=1, padding=0, bias=False)
    rng3 = t.Generator().manual_seed(0)
    noise_for_layer = ex1_dcgan_noise_batch(2, 100, rng3)
    out = first_layer(noise_for_layer)
    assert out.shape == (2, 512, 4, 4), f'first layer output wrong: {tuple(out.shape)}'

    # Different latent_dim works.
    rng4 = t.Generator().manual_seed(0)
    small_lat = ex1_dcgan_noise_batch(4, 16, rng4)
    assert small_lat.shape == (4, 16, 1, 1)

    # --- Visualization: noise heatmap (B × latent_dim, collapsing 1×1) ---
    rng_v = t.Generator().manual_seed(7)
    viz_noise = ex1_dcgan_noise_batch(16, 64, rng_v)
    viz_flat = viz_noise.squeeze(-1).squeeze(-1)   # (16, 64)
    fig, ax = plt.subplots(figsize=(8, 4))
    im = ax.imshow(viz_flat.numpy(), aspect='auto', cmap='RdBu_r', vmin=-3, vmax=3)
    ax.set_xlabel('latent dim'); ax.set_ylabel('batch idx')
    ax.set_title(f'noise batch (B=16, latent=64) — shape after squeeze: {tuple(viz_flat.shape)}')
    plt.colorbar(im, ax=ax, fraction=0.046, label='noise value')
    plt.tight_layout()
    plt.show()
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
def ex1_dcgan_noise_batch(batch_size: int, latent_dim: int, generator: 'torch.Generator') -> Tensor:
    return t.randn(batch_size, latent_dim, 1, 1, generator=generator)
```

**Why the trailing 1×1.** The DCGAN generator is fully convolutional. Its first `ConvTranspose2d(latent_dim, ..., kernel_size=4, stride=1, padding=0)` blows a `1×1` seed up to `4×4` (output shape = `(1-1)*1 - 2*0 + 4 = 4`). Without the spatial prefix the conv has no spatial axis to grow from.

**Why `t.randn`, not `t.rand`.** Standard normal vs uniform — different priors give different sample distributions. DCGAN's prior is `N(0, I)`; flipping to `Uniform(0, 1)` will train but produces visibly worse samples and mismatched latent-space interpolation behavior.

**`generator=` for reproducibility.** Passing an explicit `torch.Generator` lets the caller control the RNG without touching the global state. Critical for unit tests, training reproducibility, and seeding multi-GPU jobs.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()